# AUFS-DyClust Quickstart

End-to-end clustering dengan **AUFS-Samba (Engine C + auto-K)** untuk dataset mixed-type.

| | |
|---|---|
| Package | [mixclust](https://github.com/hqers/mixclust) |
| Tools | [hqers.my.id/mixclust](https://hqers.my.id/mixclust/) |
| Paper | [AUFS-Samba — IEEE ACCESS 2026](https://doi.org/10.1109/ACCESS.2026.3653624) |


## 0. Install / Update MixClust

In [ ]:
import sys, subprocess

GITHUB_URL  = 'git+https://github.com/hqers/mixclust.git'
MIN_VERSION = '1.0.7'  # update jika fitur baru diperlukan

def _ver(v):
    try: return tuple(int(x) for x in v.split('.'))
    except: return (0,)

# Cek numpy/pyarrow conflict (umum di shared JupyterHub)
try:
    import pyarrow
except Exception:
    print('numpy/pyarrow conflict — downgrade numpy<2...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           '-q', '--user', 'numpy<2.0'])
    print('RESTART KERNEL lalu jalankan ulang cell ini.')
    raise SystemExit('Restart kernel diperlukan')

needs_install = False
try:
    import mixclust as _mc
    if _ver(_mc.__version__) < _ver(MIN_VERSION):
        print(f'Versi {_mc.__version__} < min {MIN_VERSION} — reinstall...')
        needs_install = True
    else:
        print(f'mixclust {_mc.__version__} ok (min: {MIN_VERSION})')
except ImportError:
    print('mixclust belum terinstall — install sekarang...')
    needs_install = True

if needs_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           '-q', '--user',
                           '--upgrade-strategy', 'only-if-needed',
                           GITHUB_URL])
    import importlib, mixclust
    importlib.reload(mixclust)
    print(f'mixclust {mixclust.__version__} berhasil diinstall')
else:
    import mixclust

print(f'mixclust versi: {mixclust.__version__}')


## 1. Setup & Load Data

In [ ]:
import pandas as pd
from pathlib import Path
from mixclust import run_generic_end2end, AUFSParams

DATA_PATH  = Path('data/HH_CLUSTERING_ready.csv')  # ganti sesuai dataset
SEP        = ';'                                    # separator CSV
ID_COL     = 'HHID'                                 # kolom ID, atau None
LABEL_COL  = None                                   # kolom label (dibuang dari fitur)
DROP_COLS  = []                                     # kolom tambahan yang dieksklusi
OUTDIR     = Path('output/')

OUTDIR.mkdir(parents=True, exist_ok=True)

# Hapus redundancy cache lama jika ganti dataset
import os
cache_f = 'cache/redundancy_k5.pkl'
if os.path.exists(cache_f):
    os.remove(cache_f)
    print(f'Cache lama dihapus: {cache_f}')

df = pd.read_csv(DATA_PATH, sep=SEP)
print(f'Loaded: {len(df):,} rows × {df.shape[1]} columns')
df.head(3)


## 2. Data Quality Check

In [ ]:
import numpy as np

exclude   = [c for c in ([LABEL_COL] + DROP_COLS) if c and c in df.columns]
df_ready  = df.drop(columns=exclude, errors='ignore').copy()

zero_frac = (df_ready == 0).mean()
null_frac = df_ready.isnull().mean()

ZERO_THRESH = 0.80  # drop jika >80% nol
NULL_THRESH = 0.30  # drop jika >30% missing

drop_zero = zero_frac[zero_frac > ZERO_THRESH].index.tolist()
drop_null = null_frac[null_frac > NULL_THRESH].index.tolist()
drop_auto = sorted(set(drop_zero + drop_null))

print(f'Shape awal : {df_ready.shape}')
print(f'\n{"Kolom":30s} {"%Nol":>8s}  {"%Missing":>8s}  {"Status":>10s}')
print('-' * 62)
for col in df_ready.columns:
    z, n = zero_frac[col], null_frac[col]
    flag = '⚠ DROP' if col in drop_auto else 'ok'
    if z > 0.30 or n > 0.05 or col in drop_auto:
        print(f'  {col:28s} {z:>7.1%}   {n:>7.1%}   {flag}')

if drop_auto:
    print(f'\nAuto-drop ({len(drop_auto)} kolom): {drop_auto}')
    df_ready = df_ready.drop(columns=drop_auto)
    print(f'Shape setelah drop: {df_ready.shape}')
else:
    print('\n✓ Tidak ada kolom yang perlu di-drop')

# Edit manual jika perlu:
# EXTRA_DROP = ['kolom_a']
# df_ready = df_ready.drop(columns=EXTRA_DROP, errors='ignore')

print(f'\nFitur final: {df_ready.shape[1]} kolom, {len(df_ready):,} baris')


## 3. Konfigurasi Pipeline

In [ ]:
params = AUFSParams(
    engine_mode       = 'C',
    auto_k            = True,
    c_min             = 3,
    c_max             = 7,
    sa_iters          = 50,
    mab_T             = 12,
    mab_k             = 6,
    auto_reward       = True,   # Engine C pilih otomatis: SS Gower jika n≤2000, L-Sil jika n>2000
    hac_mode          = 'hybrid',
    cluster_adapter_lambda = 0.6,
    run_structural_control = True,
    sc_lnc_threshold  = 0.5,
    enable_screening  = True,
    screening_k_values = (3, 4, 5, 6, 7),
    screening_prune_threshold = 0.20,
    auto_algorithms   = ['kprototypes', 'hac_gower'],
    sa_neighbor_mode  = 'full',
    sa_min_size       = 3,
    kmsnc_k           = 5,
    build_redundancy_parallel = True,
    build_redundancy_cache = 'cache/redundancy_k5.pkl',
    mab_n_jobs        = 2,
    red_backend       = 'loky',
    lsil_topk         = 1,
    per_cluster_proto_if_many = 1,
    lsil_proto_sample_cap = 200,
    ss_max_n          = 2000,
    # dav_anchor_cols = ['kolom_anchor_1', 'kolom_anchor_2'],  # aktifkan untuk DAV
    random_state      = 42,
    verbose           = True,
    show_progress     = True,
)


## 4. Jalankan Pipeline

In [ ]:
res = run_generic_end2end(
    df_ready      = df_ready,
    outdir        = str(OUTDIR),
    id_col        = ID_COL,
    drop_cols     = exclude,
    params        = params,
    n_clusters_hint = 3,
    verbose       = True,
)
res


## 5. Verifikasi Hasil

In [ ]:
import json

m = json.loads((OUTDIR / 'metrics_internal.json').read_text())
print(f'K final    : {m.get("best_K")}')
print(f'Algoritma  : {m.get("final_algo")}')
print(f'Reward     : {m["best_reward"]:.4f}' if m.get('best_reward') is not None else 'Reward     : n/a')
print(f'SS Gower   : {m["final_ss_gower"]:.4f}' if m.get('final_ss_gower') is not None else 'SS Gower   : n/a')
sc = m.get('structural_control') or {}
print(f'LNC*       : {sc.get("lnc_score", "n/a")}')
print(f'Runtime    : {m.get("timing_s",{}).get("total_s",0):.1f}s')

assign = pd.read_csv(OUTDIR / 'cluster_assignments.csv')
print(f'\nDistribusi klaster:')
print(assign['cluster'].value_counts().sort_index())


## 6. Cluster Size Chart

In [ ]:
import matplotlib.pyplot as plt

assign = pd.read_csv(OUTDIR / 'cluster_assignments.csv')
vals, counts = np.unique(assign['cluster'], return_counts=True)

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(vals.astype(str), counts, color='#4f8ef7', edgecolor='#1c2030')
ax.bar_label(bars, fmt=lambda x: f'{int(x):,}')
ax.set_xlabel('Cluster'); ax.set_ylabel('N baris')
ax.set_title(f'MixClust — K={len(vals)}')
plt.tight_layout()
plt.savefig(OUTDIR / 'cluster_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTDIR}/cluster_dist.png')


## 7. Profiles

In [ ]:
# Baca profiles.md — ringkasan tiap klaster
profiles_md = (OUTDIR / 'profiles.md').read_text()
print(profiles_md)

# Tampilkan profiles_table
pd.read_csv(OUTDIR / 'profiles_table.csv').head(20)


## 8. Interpretasi

Output yang dihasilkan:

| File | Isi |
|---|---|
| `features.csv` | Fitur terpilih oleh AUFS-Samba |
| `cluster_assignments.csv` | ID + label klaster per baris |
| `profiles.json` | Statistik lengkap (mean, median, lift, chi2, ANOVA) |
| `profiles_table.csv` | Tabel ringkas per klaster |
| `profiles.md` | Narasi singkat tiap klaster |
| `metrics_internal.json` | K final, reward, SS Gower, runtime |
| `config.json` | Parameter lengkap yang dipakai |
| `summary.xlsx` | Satu baris per run — mudah dibandingkan antar seed |

Untuk visualisasi interaktif, upload file-file di atas ke:
**[hqers.my.id/mixclust/profiles.html](https://hqers.my.id/mixclust/profiles.html)**
